System information (for reproducibility):

In [28]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [29]:
using Pkg
Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/01-intro`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/01-intro/Project.toml`
  [6e4b80f9] BenchmarkTools v1.6.3
  [6f49c342] RCall v0.14.12
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


## Basic information

* Tue/Thu 1pm-2:50pm @ CHS 41-268.   

* Instructor: Dr. Hua Zhou.  

## What is statistics?

* Statistics, the science of *data analysis*, is the applied mathematics in the 21st century. 

* People (scientists, goverment, health professionals, companies) collect data in order to answer certain questions. Statisticians's job is to help them extract knowledge and insights from data. 

* If existing software tools readily solve the problem, all the better. 

* Often statisticians need to implement their own methods, test new algorithms, or tailor classical methods to new types of data (big, streaming). 

* This entails at least two essential skills: **programming** and fundamental knowledge of **algorithms**. 

## What is this course about?

* **Not** a course on statistical packages. It does not answer questions such as _How to fit a linear mixed model in R,  Julia, SAS, SPSS, or Stata?_

* **Not** a pure programming course, although programming is important and we do homework in Julia.  

* **Not** a course on data science. [BIOSTAT 203B (Introduction to Data Science)](https://ucla-biostat-203b.github.io/2026winter/schedule/schedule-lec1.html) in winter quarter focuses on some R tools for data scientists.

* This course focuses on **algorithms**, mostly those in **numerical linear algebra** and **numerical optimization**. 

## Learning objectives

1. Be highly appreciative of this quote by [James Gentle](https://www.google.com/books/edition/Computational_Statistics/mQ5KAAAAQBAJ?hl=en&gbpv=1&dq=inauthor:%22James+E.+Gentle%22)

    > The form of a mathematical expression and the way the expression should be evaluated in actual practice may be quite different.

    Examples: $\mathbf{X}^T \mathbf{W} \mathbf{X}$, $\operatorname{tr} (\mathbf{A} \mathbf{B})$, $\operatorname{diag} (\mathbf{A} \mathbf{B})$, multivariate normal density, ...  

2. Become **memory-conscious**. You care about looping order. You do benchmarking on hot functions fanatically to make sure it's not allocating.    

3. **No inversion mentality**. Whenever you see a matrix inverse in mathematical expression, your brain reacts with *matrix decomposition*, *iterative solvers*, etc. For R users, that means you almost never use the `solve(M)` function to obtain inverse of a matrix $\boldsymbol{M}$.   

    Examples: $(\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$, $\mathbf{y}^T \boldsymbol{\Sigma}^{-1} \mathbf{y}$, Newton-Raphson algorithm, ...   

4. Master some basic strategies to solve **big data** problems. 

    Examples: how Google solve the PageRank problem with $10^{9}$ webpages, linear regression with $10^7$ observations, etc.  

5. No afraid of **optimizations** and treat it as a technology. Be able to recognize some major optimization problem classes and choose the best solver(s) correspondingly.

6. Be immune to the language fight. 

## Course logistics

* Course webpage: <https://ucla-biostat-257.github.io/2026spring>.

* [Syllabus](https://ucla-biostat-257.github.io/2026spring/syllabus/syllabus.html).

* Check the [Schedule](https://ucla-biostat-257.github.io/2026spring/schedule/schedule.html) page frequently. 

* Jupyter notebooks will be posted/updated before each lecture.

## How to get started

All course materials are in GitHub repo <https://github.com/ucla-biostat-257/2026spring>. Lecture notes are Jupyter Notebooks (`.ipynb` files) in the `slides` folder. It is a good idea to learn by running through the code examples. You can do this in several ways. 

### Run Jupyter Notebook in Binder

A quick and easy way to run the Jupyter Notebooks is Binder, a free service that allows users to run Jupyter Notebooks in cloud. Simply follow the Binder link at the [schedule](https://ucla-biostat-257.github.io/2026spring/schedule/schedule.html) page. 

If you want the JupyterLab interface, replace the `tree` by `lab` in the URL.  

### Run Jupyter Notebook locally on your own computer

1. Install Julia v1.12.x following instructions at <https://julialang.org/downloads/>.

2. Install `IJulia` package. Open Julia REPL, type `]` to enter the package mode, then type
```julia
add IJulia
build IJulia
```

3. Git clone the course material.   
```bash
git clone https://github.com/ucla-biostat-257/2026spring.git biostat-257-2026spring
```
You can change `biostat-257-2026spring` to any other directory name you prefer.

4. On terminal, enter the folder for the ipynb file you want to run, e.g. `biostat-257-2026spring/slides/01-intro/`. 
5. Open Julia REPL, type  
```julia  
using IJulia
jupyterlab(dir = pwd())
```
to open the JupyterLab in browser or
```julia  
using IJulia
notebook(dir = pwd())
```
to open a Jupyter notebook.

6. Course material is updated frequently. Remember to `git pull` to obtain the most recent material.

### Run Jupyter Notebook in Positron (or VS Code)

1. Install [Julia](https://julialang.org/downloads/), [Positron](https://positron.posit.co/) (or [VS Code](https://code.visualstudio.com/)), and [Quarto](https://quarto.org/docs/get-started/).

2. Open Positron (or VS Code) and install extensions: Julia, Jupyter, Quarto, GitHub Copilot.

3. Git clone the course material.   
```bash
git clone https://github.com/ucla-biostat-257/2026spring.git biostat-257-2026spring
```
You can change `biostat-257-2026spring` to any other directory name you prefer.

4. Open the folder in Positron (or VS Code).

## In class dicussion

The logistic regression is typically estimated by the Fisher scoring algorithm, or iteratively reweighted least squares (IWLS), which iterates according to
$$
\boldsymbol{\beta}^{(t)} = (\mathbf{X}^T \mathbf{W}^{(t)} \mathbf{X})^{-1} \mathbf{X}^T \mathbf{W}^{(t)} \mathbf{z}^{(t)},
$$
where $\mathbf{z}^{(t)}$ are pseudo-responses and $\mathbf{W}^{(t)} = \text{diag}(\mathbf{w}^{(t)})$ is a diagonal matrix with nonnegative weights on the diagonal. Superscript $t$ is the iterate number.

### Poll: Numeric

How much speedup can we achieve, by careful consideration of flops and memory usage, over the following naive implementation?
```julia
inv(X' * diagm(w) * X) * X' * diagm(w) * z
```

### Experiment

First generate some data.

In [30]:
using LinearAlgebra, Random

# Random seed for reproducibility
Random.seed!(257)
# samples, features
n, p = 5000, 100
# design matrix
X = [ones(n) randn(n, p - 1)]
# pseudo-responses
z = randn(n)
# weights
w = 0.25 * rand(n);

### Method 1

The following code literally translates the mathematical expression into code.

In [31]:
# method 1 
res1 = inv(X' * diagm(w) * X) * X' * diagm(w) * z

100-element Vector{Float64}:
 -0.0047313526500880445
  0.009183070405469698
 -0.016275221473477954
 -0.013279497350630196
  0.020014830435187928
  0.020674778392632622
  0.0007810692137151214
 -0.012360822702514545
  0.0011239267098812184
  0.01169028835045102
  ⋮
  0.0017666315860712682
  0.018897291502571356
 -0.02628676655057106
  0.03492841833693639
  0.008008587435710198
  0.008244324612943884
  0.013637070959968484
  0.01360393323312991
 -0.00539638287983003

In [32]:
using BenchmarkTools

bm1 = @benchmark ((inv($X' * diagm($w) * $X) * $X') * diagm($w)) * $z
bm1

BenchmarkTools.Trial: 107 samples with 1 evaluation per sample.
 Range (min … max):  35.775 ms … 134.593 ms  ┊ GC (min … max): 1.12% … 56.23%
 Time  (median):     40.653 ms               ┊ GC (median):    2.16%
 Time  (mean ± σ):   46.944 ms ±  15.285 ms  ┊ GC (mean ± σ):  9.13% ± 12.35%

  █▃ ▂                                                          
  ██▅█▃▅▃▅▃▃▅▃▁▃▃▃▄▃▆▁▃▃▃▃▃▁▃▁▃▃▃▁▁▁▁▃▁▁▃▁▁▁▁▁▁▃▁▁▃▁▁▁▃▃▁▁▁▁▁▃ ▃
  35.8 ms         Histogram: frequency by time           94 ms <

 Memory estimate: 393.21 MiB, allocs estimate: 29.

Several unwise choices of algorithms waste lots of flops. The memeory allocations, caused by intermediate results, also slow down the program because of the need for garbage collection. This is a common mistake a beginner programmer in a high-level language makes. For example, the following R code (same algorithm on the same data) shows similar allocation. R code is slower than Julia possibly because of the outdated BLAS/LAPACK library being used. 

In [33]:
using RCall

R"sessionInfo()"

RObject{VecSxp}
R version 4.5.2 (2025-10-31)
Platform: aarch64-apple-darwin20
Running under: macOS Tahoe 26.4

Matrix products: default
BLAS:   /System/Library/Frameworks/Accelerate.framework/Versions/A/Frameworks/vecLib.framework/Versions/A/libBLAS.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8

time zone: America/Los_Angeles
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] bench_1.1.4

loaded via a namespace (and not attached):
 [1] compiler_4.5.2  magrittr_2.0.4  cli_3.6.5       pillar_1.11.1  
 [5] glue_1.8.0      tibble_3.3.1    utf8_1.2.6      vctrs_0.6.5    
 [9] lifecycle_1.0.5 pkgconfig_2.0.3 rlang_1.1.7     profmem_0.7.0  


In [34]:
R"""
library(bench)

# Interpolate Julia variables into R workspace
X <- $X
w <- $w
z <- $z

rbm1 <- bench::mark(
  solve(t(X) %*% diag(w) %*% X) %*% t(X) %*% diag(w) %*% z,
  iterations = 10
  ) |> 
  print(width = Inf)
""";

# A tibble: 1 × 13
  expression                                                    min   median
  <bch:expr>                                               <bch:tm> <bch:tm>
1 solve(t(X) %*% diag(w) %*% X) %*% t(X) %*% diag(w) %*% z   58.1ms   73.8ms
  `itr/sec` mem_alloc `gc/sec` n_itr  n_gc total_time result         
      <dbl> <bch:byt>    <dbl> <int> <dbl>   <bch:tm> <list>         
1      13.5     401MB     27.0    10    20      741ms <dbl [100 × 1]>
  memory              time            gc               
  <list>              <list>          <list>           
1 <Rprofmem [14 × 3]> <bench_tm [10]> <tibble [10 × 3]>


┌ Warning: RCall.jl: Warning: Some expressions had a GC in every iteration; so filtering is disabled.
└ @ RCall /Users/huazhou/.julia/packages/RCall/FKyW5/src/io.jl:166


### Method 2

In the following code, we make smarter choice of algorithms (rearranging order of evaluation; utilizing data structures such as diagonal matrix, triangular matrix, and positive definite matrices) and get rid of memeory allocation by pre-allocating intermediate arrays. 

In [35]:
# preallocation
XtWt = Matrix{Float64}(undef, p, n)
XtX = Matrix{Float64}(undef, p, p)
Xtz = Vector{Float64}(undef, p)

function myfun(X, z, w, XtWt, XtX, Xtz)
    # XtWt = X' * W
    mul!(XtWt, transpose(X), Diagonal(w))
    # XtX = X' * W * X
    mul!(XtX, XtWt, X)
    # Xtz = X' * W * z
    mul!(Xtz, XtWt, z)
    # Cholesky on XtX
    LAPACK.potrf!('U', XtX)
    # Two triangular solves to solve (XtX) \ (Xtz)
    BLAS.trsv!('U', 'T', 'N', XtX, Xtz)
    BLAS.trsv!('U', 'N', 'N', XtX, Xtz)
end

# First check correctness vs Method 1
res2 = myfun(X, z, w, XtWt, XtX, Xtz)

100-element Vector{Float64}:
 -0.004731352650088043
  0.009183070405469736
 -0.016275221473477902
 -0.013279497350630177
  0.020014830435187862
  0.020674778392632622
  0.0007810692137151314
 -0.012360822702514545
  0.001123926709881188
  0.011690288350451031
  ⋮
  0.0017666315860712875
  0.018897291502571387
 -0.026286766550571057
  0.034928418336936204
  0.008008587435710167
  0.008244324612943875
  0.013637070959968462
  0.013603933233129917
 -0.005396382879830023

In [36]:
bm2 = @benchmark myfun($X, $z, $w, $XtWt, $XtX, $Xtz)
bm2

BenchmarkTools.Trial: 4305 samples with 1 evaluation per sample.
 Range (min … max):  744.375 μs … 47.365 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     837.500 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.160 ms ±  1.122 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▃▂▂▁▂▂▂▂▁▁▁▁▁▁ ▁                                            
  ██████████████████████▇█▆▇▇▆▆▄▅▅▅▅▅▅▆▃▄▄▃▅▅▄▄▃▄▅▂▃▄▂▄▃▂▄▄▄▅▄ █
  744 μs        Histogram: log(frequency) by time      4.42 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In R, a better implementation is
```r
solve(t(X * w) %*% X, t(X) %*% (z * w))
```
It's much faster than the naive implementation. To achieve zero memory allocation, some low-level coding using C++ and RcppEigen is necessary.

In [37]:
R"""
rbm2 <- bench::mark(
  solve(t(X * w) %*% X, t(X) %*% (z * w)),
  ) |> 
  print(width = Inf)
""";

# A tibble: 1 × 13
  expression                                   min   median `itr/sec` mem_alloc
  <bch:expr>                              <bch:tm> <bch:tm>     <dbl> <bch:byt>
1 solve(t(X * w) %*% X, t(X) %*% (z * w))   2.46ms   2.71ms      356.    11.6MB
  `gc/sec` n_itr  n_gc total_time result          memory             
     <dbl> <int> <dbl>   <bch:tm> <list>          <list>             
1     49.6   151    21      424ms <dbl [100 × 1]> <Rprofmem [10 × 3]>
  time             gc                
  <list>           <list>            
1 <bench_tm [172]> <tibble [172 × 3]>


### Conclusion

By careful consideration of the computational algorithms, we achieve a whooping speedup (in Julia) of

In [38]:
# speed-up
median(bm1.times) / median(bm2.times)

48.54089552238806

For PhD students, that means, instead of waiting more than two months for your simulations to finish, you only need one day!